| i | θᵢ | dᵢ (mm) | aᵢ (mm) | αᵢ |
|---:|---|---:|---:|---:|
| 1 | θ₁ | 134.75 | 0 | +90° |
| 2 | θ₂ − 90° | 0 | -110 | 0° |
| 3 | θ₃ | 0 | -96 | 0° |
| 4 | θ₄ − 90° | 63.4 | 0 | +90° |
| 5 | θ₅ + 90° | 75.05 | 0 | -90° |
| 6 | θ₆ | 50 | 0 | 0° |

In [11]:
from pymycobot.mycobot import MyCobot

PORT = "/dev/ttyUSB0"
BAUD = 1000000

mc = MyCobot(PORT, BAUD)

print("Conexión creada")
print("Ángulos actuales:", mc.get_angles())

Conexión creada
Ángulos actuales: [0.35, 0.0, 0.26, -2.28, -0.17, -12.21]


In [9]:
import numpy as np


# ==========================================
# MATRIZ DENAVIT-HARTENBERG
# ==========================================

def dh_matrix(theta, d, a, alpha):

    theta = np.deg2rad(theta)
    alpha = np.deg2rad(alpha)

    return np.array([
        [
            np.cos(theta),
            -np.sin(theta) * np.cos(alpha),
            np.sin(theta) * np.sin(alpha),
            a * np.cos(theta)
        ],
        [
            np.sin(theta),
            np.cos(theta) * np.cos(alpha),
            -np.cos(theta) * np.sin(alpha),
            a * np.sin(theta)
        ],
        [
            0,
            np.sin(alpha),
            np.cos(alpha),
            d
        ],
        [
            0,
            0,
            0,
            1
        ]
    ])


# ==========================================
# CINEMÁTICA DIRECTA
# ==========================================

def fk(theta):

    # Ángulos físicos entregados por get_angles()
    t1, t2, t3, t4, t5, t6 = theta

    # Offsets de la tabla DH
    q1 = t1
    q2 = t2 - 90
    q3 = t3
    q4 = t4 - 90
    q5 = t5 + 90
    q6 = t6

    # Tabla DH
    A1 = dh_matrix(q1, 134.75, 0,    90)
    A2 = dh_matrix(q2, 0,     -110,   0)
    A3 = dh_matrix(q3, 0,      -96,   0)
    A4 = dh_matrix(q4, 63.4,     0,  90)
    A5 = dh_matrix(q5, 75.05,    0, -90)
    A6 = dh_matrix(q6, 50,       0,   0)

    T06 = A1 @ A2 @ A3 @ A4 @ A5 @ A6

    return T06


# ==========================================
# 1. LEER ÁNGULOS REALES DEL JETCOBOT
# ==========================================

q = mc.get_angles()

print("\n================================")
print("ÁNGULOS REALES DEL JETCOBOT")
print("================================")

print(q)


# ==========================================
# 2. CALCULAR FK
# ==========================================

T = fk(q)

x_pred = T[0, 3]
y_pred = T[1, 3]
z_pred = T[2, 3]


print("\n================================")
print("MATRIZ T06")
print("================================")

print(np.round(T, 3))


print("\n================================")
print("COORDENADAS CALCULADAS POR FK")
print("================================")

print(f"X = {x_pred:.2f} mm")
print(f"Y = {y_pred:.2f} mm")
print(f"Z = {z_pred:.2f} mm")


# ==========================================
# 3. LEER COORDENADAS DEL JETCOBOT
# ==========================================

coords = mc.get_coords()

print("\n================================")
print("COORDENADAS DEL JETCOBOT")
print("================================")

print(coords)


if coords is not None and len(coords) >= 3:

    x_real = coords[0]
    y_real = coords[1]
    z_real = coords[2]


    print("\nPosición JetCobot:")

    print(f"X = {x_real:.2f} mm")
    print(f"Y = {y_real:.2f} mm")
    print(f"Z = {z_real:.2f} mm")


    # ==========================================
    # 4. CALCULAR ERROR POR EJE
    # ==========================================

    error_x = abs(x_real - x_pred)
    error_y = abs(y_real - y_pred)
    error_z = abs(z_real - z_pred)


    # ==========================================
    # 5. ERROR CARTESIANO TOTAL
    # ==========================================

    error_total = np.sqrt(
        (x_real - x_pred) ** 2 +
        (y_real - y_pred) ** 2 +
        (z_real - z_pred) ** 2
    )


    # ==========================================
    # 6. MOSTRAR COMPARACIÓN
    # ==========================================

    print("\n================================")
    print("COMPARACIÓN FK VS JETCOBOT")
    print("================================")

    print(
        f"FK:       "
        f"({x_pred:.2f}, "
        f"{y_pred:.2f}, "
        f"{z_pred:.2f}) mm"
    )

    print(
        f"JetCobot: "
        f"({x_real:.2f}, "
        f"{y_real:.2f}, "
        f"{z_real:.2f}) mm"
    )




ÁNGULOS REALES DEL JETCOBOT
[0.35, 0.0, 0.26, -1.23, -0.35, -44.29]

MATRIZ T06
[[ 1.20000e-02 -1.20000e-02  1.00000e+00  5.12150e+01]
 [-7.16000e-01 -6.98000e-01 -0.00000e+00 -6.33940e+01]
 [ 6.98000e-01 -7.16000e-01 -1.70000e-02  4.14942e+02]
 [ 0.00000e+00  0.00000e+00  0.00000e+00  1.00000e+00]]

COORDENADAS CALCULADAS POR FK
X = 51.21 mm
Y = -63.39 mm
Z = 414.94 mm

COORDENADAS DEL JETCOBOT
[49.7, -64.6, 409.9, -91.35, -44.29, -89.05]

Posición JetCobot:
X = 49.70 mm
Y = -64.60 mm
Z = 409.90 mm

COMPARACIÓN FK VS JETCOBOT
FK:       (51.21, -63.39, 414.94) mm
JetCobot: (49.70, -64.60, 409.90) mm


In [ ]:
print(mc.get_coords())

In [ ]:
import time
import numpy as np

poses = [
    [0, 0, 0, 0, 0, 0],       # HOME
    [0, 30, -30, 0, 0, 0],
    [30, 20, -40, 20, 0, 0]
]

VELOCIDAD = 5

for i, q in enumerate(poses, start=1):

    print("\n================================")
    print(f"POSE {i}")
    print("================================")

    # ==========================================
    # 1. PREDICCIÓN ANTES DE MOVER EL ROBOT
    # ==========================================

    T = fk(q)

    x_pred = T[0, 3]
    y_pred = T[1, 3]
    z_pred = T[2, 3]

    print("\nÁngulos enviados:")
    print(q)

    print("\nCOORDENADAS PREDICHAS POR FK:")
    print(f"X = {x_pred:.2f} mm")
    print(f"Y = {y_pred:.2f} mm")
    print(f"Z = {z_pred:.2f} mm")

    # ==========================================
    # 2. MOVER JETCOBOT
    # ==========================================

    print("\nMoviendo JetCobot...")

    mc.send_angles(q, VELOCIDAD)

    time.sleep(8)

    # ==========================================
    # 3. LEER ÁNGULOS REALES
    # ==========================================

    q_real = mc.get_angles()

    print("\nÁngulos reales ejecutados:")
    print(q_real)

    # ==========================================
    # 4. LEER COORDENADAS DEL JETCOBOT
    # ==========================================

    coords = mc.get_coords()

    print("\nCoordenadas completas del JetCobot:")
    print(coords)

    if coords is None or len(coords) < 3:
        print("No se pudieron obtener las coordenadas.")
        continue

    x_real = coords[0]
    y_real = coords[1]
    z_real = coords[2]

    print("\nCOORDENADAS DEL JETCOBOT:")
    print(f"X = {x_real:.2f} mm")
    print(f"Y = {y_real:.2f} mm")
    print(f"Z = {z_real:.2f} mm")

    # ==========================================
    # 5. ERROR CARTESIANO
    # ==========================================

    error_x = abs(x_real - x_pred)
    error_y = abs(y_real - y_pred)
    error_z = abs(z_real - z_pred)

    error_total = np.sqrt(
        (x_real - x_pred) ** 2 +
        (y_real - y_pred) ** 2 +
        (z_real - z_pred) ** 2
    )

    # ==========================================
    # 6. RESULTADOS
    # ==========================================

    print("\n------------- COMPARACIÓN -------------")

    print(
        f"FK:       "
        f"({x_pred:.2f}, "
        f"{y_pred:.2f}, "
        f"{z_pred:.2f}) mm"
    )

    print(
        f"JetCobot: "
        f"({x_real:.2f}, "
        f"{y_real:.2f}, "
        f"{z_real:.2f}) mm"
    )


    print("================================")

# Item 4

In [6]:
print("Ángulos:", mc.get_angles())
print("Coordenadas:", mc.get_coords())
print("¿Está moviéndose?:", mc.is_moving())

Ángulos: [0.35, 0.0, 0.26, -1.23, -0.35, -44.29]
Coordenadas: [49.7, -64.6, 409.9, -91.35, -44.29, -89.05]
¿Está moviéndose?: 0


# MEjorado item4

In [26]:
from pymycobot.mycobot import MyCobot

PORT = "/dev/ttyUSB0"
BAUD = 1000000

mc = MyCobot(PORT, BAUD)

print("Conexión creada")
print("Ángulos actuales:", mc.get_angles())

Conexión creada
Ángulos actuales: [-1.05, -15.9, 35.5, -19.07, 0.08, 1.49]


In [27]:
import numpy as np


# ==========================================
# MATRIZ DENAVIT-HARTENBERG
# ==========================================

def dh_matrix(theta, d, a, alpha):

    theta = np.deg2rad(theta)
    alpha = np.deg2rad(alpha)

    return np.array([
        [
            np.cos(theta),
            -np.sin(theta) * np.cos(alpha),
            np.sin(theta) * np.sin(alpha),
            a * np.cos(theta)
        ],
        [
            np.sin(theta),
            np.cos(theta) * np.cos(alpha),
            -np.cos(theta) * np.sin(alpha),
            a * np.sin(theta)
        ],
        [
            0,
            np.sin(alpha),
            np.cos(alpha),
            d
        ],
        [
            0,
            0,
            0,
            1
        ]
    ])


# ==========================================
# CINEMÁTICA DIRECTA
# ==========================================

def fk(theta):

    # Ángulos físicos entregados por get_angles()
    t1, t2, t3, t4, t5, t6 = theta

    # Offsets de la tabla DH
    q1 = t1
    q2 = t2 - 90
    q3 = t3
    q4 = t4 - 90
    q5 = t5 + 90
    q6 = t6

    # Tabla DH
    A1 = dh_matrix(q1, 134.75, 0,    90)
    A2 = dh_matrix(q2, 0,     -110,   0)
    A3 = dh_matrix(q3, 0,      -96,   0)
    A4 = dh_matrix(q4, 63.4,     0,  90)
    A5 = dh_matrix(q5, 75.05,    0, -90)
    A6 = dh_matrix(q6, 50,       0,   0)

    T06 = A1 @ A2 @ A3 @ A4 @ A5 @ A6

    return T06


# ==========================================
# 1. LEER ÁNGULOS REALES DEL JETCOBOT
# ==========================================

q = mc.get_angles()

print("\n================================")
print("ÁNGULOS REALES DEL JETCOBOT")
print("================================")

print(q)


# ==========================================
# 2. CALCULAR FK
# ==========================================

T = fk(q)

x_pred = T[0, 3]
y_pred = T[1, 3]
z_pred = T[2, 3]


print("\n================================")
print("MATRIZ T06")
print("================================")

print(np.round(T, 3))


print("\n================================")
print("COORDENADAS CALCULADAS POR FK")
print("================================")

print(f"X = {x_pred:.2f} mm")
print(f"Y = {y_pred:.2f} mm")
print(f"Z = {z_pred:.2f} mm")


# ==========================================
# 3. LEER COORDENADAS DEL JETCOBOT
# ==========================================

coords = mc.get_coords()

print("\n================================")
print("COORDENADAS DEL JETCOBOT")
print("================================")

print(coords)


if coords is not None and len(coords) >= 3:

    x_real = coords[0]
    y_real = coords[1]
    z_real = coords[2]


    print("\nPosición JetCobot:")

    print(f"X = {x_real:.2f} mm")
    print(f"Y = {y_real:.2f} mm")
    print(f"Z = {z_real:.2f} mm")


    # ==========================================
    # 4. CALCULAR ERROR POR EJE
    # ==========================================

    error_x = abs(x_real - x_pred)
    error_y = abs(y_real - y_pred)
    error_z = abs(z_real - z_pred)


    # ==========================================
    # 5. ERROR CARTESIANO TOTAL
    # ==========================================

    error_total = np.sqrt(
        (x_real - x_pred) ** 2 +
        (y_real - y_pred) ** 2 +
        (z_real - z_pred) ** 2
    )


    # ==========================================
    # 6. MOSTRAR COMPARACIÓN
    # ==========================================

    print("\n================================")
    print("COMPARACIÓN FK VS JETCOBOT")
    print("================================")

    print(
        f"FK:       "
        f"({x_pred:.2f}, "
        f"{y_pred:.2f}, "
        f"{z_pred:.2f}) mm"
    )

    print(
        f"JetCobot: "
        f"({x_real:.2f}, "
        f"{y_real:.2f}, "
        f"{z_real:.2f}) mm"
    )




ÁNGULOS REALES DEL JETCOBOT
[-1.05, -15.9, 35.5, -19.07, 0.08, 1.49]

MATRIZ T06
[[-1.70000e-02  1.00000e-02  1.00000e+00  4.60670e+01]
 [-1.00000e+00  2.60000e-02 -1.70000e-02 -6.41850e+01]
 [-2.60000e-02 -1.00000e+00  9.00000e-03  4.06488e+02]
 [ 0.00000e+00  0.00000e+00  0.00000e+00  1.00000e+00]]

COORDENADAS CALCULADAS POR FK
X = 46.07 mm
Y = -64.19 mm
Z = 406.49 mm

COORDENADAS DEL JETCOBOT
[44.7, -65.3, 401.4, -89.47, 1.49, -90.95]

Posición JetCobot:
X = 44.70 mm
Y = -65.30 mm
Z = 401.40 mm

COMPARACIÓN FK VS JETCOBOT
FK:       (46.07, -64.19, 406.49) mm
JetCobot: (44.70, -65.30, 401.40) mm


In [17]:
print("Ángulos:", mc.get_angles())
print("Coordenadas:", mc.get_coords())

Ángulos: [-36.91, 0.96, -32.51, -17.31, -12.04, -25.13]
Coordenadas: [62.8, -140.7, 335.7, -144.32, -24.93, -114.44]


# MEjroado con hipotesis

In [28]:
##### import time
import numpy as np

# Requiere que ya hayas ejecutado las celdas que crean mc y definen fk().
VELOCIDAD = 5
MODO = 0                     # 0 = trayectoria angular
TIEMPO_MAXIMO = 60           # segundos
TOLERANCIA_LLEGADA = 10.0     # mm, para verificar get_coords()


def leer_coords_validas():
    """Devuelve [x, y, z, rx, ry, rz] o informa un fallo de lectura."""
    coords = mc.get_coords()

    if coords is None or len(coords) < 6:
        raise RuntimeError(f"Lectura inválida de get_coords(): {coords}")

    coords = np.asarray(coords[:6], dtype=float)

    if not np.all(np.isfinite(coords)):
        raise RuntimeError(f"Coordenadas no finitas: {coords}")

    return coords


def leer_angulos_validos():
    """Devuelve los seis ángulos reportados por el robot."""
    angulos = mc.get_angles()

    if angulos is None or len(angulos) != 6:
        raise RuntimeError(f"Lectura inválida de get_angles(): {angulos}")

    angulos = np.asarray(angulos, dtype=float)

    if not np.all(np.isfinite(angulos)):
        raise RuntimeError(f"Ángulos no finitos: {angulos}")

    return angulos


# ============================================================
# 1. LEER EL ESTADO INICIAL
# ============================================================

coords_iniciales = leer_coords_validas()
q_inicial = leer_angulos_validos()

print("\n========== ESTADO INICIAL ==========")
print("Coordenadas:", np.round(coords_iniciales, 2).tolist())
print("Ángulos:", np.round(q_inicial, 2).tolist())

p_fk_inicial = fk(q_inicial)[:3, 3]
diferencia_inicial = np.linalg.norm(
    p_fk_inicial - coords_iniciales[:3]
)

print("FK inicial [mm]:", np.round(p_fk_inicial, 2).tolist())
print(
    f"Diferencia inicial entre FK y firmware: "
    f"{diferencia_inicial:.2f} mm"
)


# ============================================================
# 2. INGRESAR EL OBJETIVO CARTESIANO
# ============================================================

print("\nIngresa coordenadas ABSOLUTAS respecto a la base.")
x_obj = float(input("X objetivo (mm): "))
y_obj = float(input("Y objetivo (mm): "))
z_obj = float(input("Z objetivo (mm): "))

p_obj = np.array([x_obj, y_obj, z_obj], dtype=float)

if not np.all(np.isfinite(p_obj)):
    raise ValueError("El objetivo contiene un valor no válido.")

# Se conserva la orientación actual del efector.
objetivo = [
    x_obj,
    y_obj,
    z_obj,
    float(coords_iniciales[3]),
    float(coords_iniciales[4]),
    float(coords_iniciales[5])
]

distancia_solicitada = np.linalg.norm(
    p_obj - coords_iniciales[:3]
)

print("\n========== OBJETIVO SOLICITADO ==========")
print("Objetivo [x, y, z, rx, ry, rz]:", objetivo)
print(f"Distancia desde el inicio: {distancia_solicitada:.2f} mm")

if distancia_solicitada < 3.0:
    raise ValueError(
        "El objetivo está demasiado cerca de la posición inicial "
        "para verificar claramente el movimiento."
    )

confirmacion = input(
    "\nVerifica el objetivo, el espacio libre y que nadie más controle "
    "el brazo. Escribe MOVER para enviar la orden: "
)

if confirmacion != "MOVER":
    print("Movimiento cancelado. No se envió ninguna orden.")

else:
    # ========================================================
    # 3. ENVIAR EL OBJETIVO AL FIRMWARE
    # ========================================================

    print("\nEnviando objetivo con send_coords()...")
    respuesta = mc.send_coords(objetivo, VELOCIDAD, MODO)
    print("Respuesta de send_coords():", respuesta)

    # La respuesta de send_coords() por sí sola NO prueba
    # que el objetivo haya sido alcanzado.
    inicio = time.monotonic()
    hubo_movimiento = False
    objetivo_observado = False
    termino_por_timeout = False
    lecturas_quieto = 0

    # ========================================================
    # 4. OBSERVAR EL MOVIMIENTO Y LA LLEGADA
    # ========================================================

    while True:
        time.sleep(0.5)
        transcurrido = time.monotonic() - inicio

        if transcurrido >= TIEMPO_MAXIMO:
            termino_por_timeout = True
            break

        try:
            coords_observadas = leer_coords_validas()
        except RuntimeError as error:
            print("No se pudo leer una muestra:", error)
            continue

        recorrido_observado = np.linalg.norm(
            coords_observadas[:3] - coords_iniciales[:3]
        )

        distancia_observada = np.linalg.norm(
            coords_observadas[:3] - p_obj
        )

        estado_movimiento = mc.is_moving()

        if recorrido_observado >= 1.0:
            hubo_movimiento = True

        if distancia_observada <= TOLERANCIA_LLEGADA:
            objetivo_observado = True

        # Solo terminar por llegada cuando hubo movimiento
        # y el robot reporta que ya se detuvo.
        if (
            hubo_movimiento
            and objetivo_observado
            and estado_movimiento == 0
        ):
            break

        # Si nunca comenzó, no hace falta esperar los 60 segundos.
        if not hubo_movimiento and transcurrido >= 10:
            break

        # Si se movió, se detuvo y sigue lejos del objetivo,
        # registrar el fallo sin esperar el tiempo máximo.
        if hubo_movimiento and estado_movimiento == 0:
            lecturas_quieto += 1

            if lecturas_quieto >= 4:
                break
        else:
            lecturas_quieto = 0

    # ========================================================
    # 5. LEER LA POSE REALMENTE EJECUTADA
    # ========================================================

    coords_finales = leer_coords_validas()
    q_real = leer_angulos_validos()
    estado_final = mc.is_moving()

    p_firmware = coords_finales[:3]
    p_fk = fk(q_real)[:3, 3]

    # ========================================================
    # 6. CALCULAR LAS TRES DISTANCIAS DE DIAGNÓSTICO
    # ========================================================

    recorrido_real = np.linalg.norm(
        p_firmware - coords_iniciales[:3]
    )

    error_firmware = np.linalg.norm(
        p_obj - p_firmware
    )

    error_fk = np.linalg.norm(
        p_obj - p_fk
    )

    diferencia_modelos = np.linalg.norm(
        p_firmware - p_fk
    )

    # ========================================================
    # 7. MOSTRAR LOS RESULTADOS
    # ========================================================

    print("\n========== RESULTADO DEL ÍTEM 4 ==========")

    print("Objetivo [mm]:",
          np.round(p_obj, 2).tolist())

    print("Posición inicial del firmware [mm]:",
          np.round(coords_iniciales[:3], 2).tolist())

    print("Posición final del firmware [mm]:",
          np.round(p_firmware, 2).tolist())

    print("Ángulos finales q [grados]:",
          np.round(q_real, 2).tolist())

    print("Posición de nuestra FK(q_real) [mm]:",
          np.round(p_fk, 2).tolist())

    print(f"\nMovimiento observado: {recorrido_real:.2f} mm")
    print(f"Objetivo vs firmware: {error_firmware:.2f} mm")
    print(f"Objetivo vs FK propia: {error_fk:.2f} mm")
    print(f"Firmware vs FK propia: {diferencia_modelos:.2f} mm")
    print("¿Sigue moviéndose?:", estado_final)

   
        # ========================================================
    # 8. MOSTRAR EL ERROR SIEMPRE Y EVALUAR LA PRUEBA
    # ========================================================
    
    print("\n========================================")
    print(f"ERROR CARTESIANO CON NUESTRA FK: {error_fk:.2f} mm")
    print(f"ERROR RESPECTO AL FIRMWARE:     {error_firmware:.2f} mm")
    print("========================================")
    
    if estado_final != 0:
        print("ESTADO: INCONCLUSO. El robot aún se mueve o no confirmó su parada.")
    
    elif recorrido_real < 1.0:
        print("ESTADO: NO VÁLIDO. No se observó movimiento.")
    
    elif error_firmware > TOLERANCIA_LLEGADA:
        print(
            "ESTADO: EL ROBOT SE MOVIÓ, PERO NO ALCANZÓ "
            "EL OBJETIVO DENTRO DE LA TOLERANCIA."
        )
    
    else:
        print("ESTADO: OBJETIVO ALCANZADO según get_coords().")
    
    if error_fk <= 10.0:
        print("Criterio de error FK ≤ 10 mm: CUMPLE")
    else:
        print("Criterio de error FK ≤ 10 mm: NO CUMPLE")


========== ESTADO INICIAL ==========
Coordenadas: [44.7, -65.3, 401.4, -89.47, 1.49, -90.95]
Ángulos: [-1.05, -15.9, 35.5, -19.07, 0.08, 1.49]
FK inicial [mm]: [46.07, -64.19, 406.49]
Diferencia inicial entre FK y firmware: 5.39 mm

Ingresa coordenadas ABSOLUTAS respecto a la base.


X objetivo (mm):  39.7
Y objetivo (mm):  -62.3
Z objetivo (mm):  393.4



========== OBJETIVO SOLICITADO ==========
Objetivo [x, y, z, rx, ry, rz]: [39.7, -62.3, 393.4, -89.47, 1.49, -90.95]
Distancia desde el inicio: 9.90 mm



Verifica el objetivo, el espacio libre y que nadie más controle el brazo. Escribe MOVER para enviar la orden:  MOVER



Enviando objetivo con send_coords()...
Respuesta de send_coords(): None

========== RESULTADO DEL ÍTEM 4 ==========
Objetivo [mm]: [39.7, -62.3, 393.4]
Posición inicial del firmware [mm]: [44.7, -65.3, 401.4]
Posición final del firmware [mm]: [36.7, -61.8, 394.8]
Ángulos finales q [grados]: [-7.55, -21.79, 47.63, -23.9, 9.58, 2.28]
Posición de nuestra FK(q_real) [mm]: [38.09, -60.61, 399.97]

Movimiento observado: 10.95 mm
Objetivo vs firmware: 3.35 mm
Objetivo vs FK propia: 6.97 mm
Firmware vs FK propia: 5.48 mm
¿Sigue moviéndose?: 0

ERROR CARTESIANO CON NUESTRA FK: 6.97 mm
ERROR RESPECTO AL FIRMWARE:     3.35 mm
ESTADO: OBJETIVO ALCANZADO según get_coords().
Criterio de error FK ≤ 10 mm: CUMPLE
